# Phase 4 v0.5 Round — Kaggle CUDA

**실행 전 체크리스트**
1. 우측 Datasets 탭 → **donghyun51/lens-phase4-v0-4** 추가 확인
2. Accelerator: **GPU T4 x2** (또는 T4 x1) 선택
3. Internet: **On** (GitHub clone 필요)

v0.5 변경: Mode3/Image 삭제, Mode1Head in_dim 384→256. 데이터는 v0.4 동일.


## Cell 1 — 환경 확인


In [ ]:
import os, subprocess, torch

print(subprocess.check_output(['nvidia-smi'], text=True))
print('torch', torch.__version__,
      '| cuda', torch.cuda.is_available(),
      '|', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'no GPU')


## Cell 2 — 데이터 경로 설정

Dataset `donghyun51/lens-phase4-v0-4`는 `/kaggle/input/lens-phase4-v0-4/`에 마운트된다.


In [ ]:
from pathlib import Path

# ── Kaggle input 고정 경로 ───────────────────────────────────────────────────
INPUT_DIR   = Path('/kaggle/input/lens-phase4-v0-4')
WORK_DIR    = Path('/kaggle/working')

TRAIN_H5    = INPUT_DIR / 'phase4_v0_4.h5'
UNFILT_H5   = INPUT_DIR / 'phase4_v0_4_eval_unfiltered.h5'
SCALER_PKL  = INPUT_DIR / 'target_scaler_phase4_v0_4.pkl'

# 파일 존재 확인
for p in [TRAIN_H5, UNFILT_H5, SCALER_PKL]:
    status = f'{p.stat().st_size/1e6:.1f} MB' if p.exists() else 'MISSING'
    print(f'{p.name}: {status}')

assert TRAIN_H5.exists(),   f'train h5 not found: {TRAIN_H5}'
assert UNFILT_H5.exists(),  f'unfiltered h5 not found: {UNFILT_H5}'
assert SCALER_PKL.exists(), f'scaler not found: {SCALER_PKL}'
print('All input files OK')


## Cell 3 — 코드 클론 (GitHub main 브랜치)


In [ ]:
import subprocess

REPO_URL  = 'https://github.com/dasbaq/GV.git'
REPO_ROOT = Path('/kaggle/working/repo')             # git clone 목적지
PROJ_DIR  = REPO_ROOT / 'Gravitational_Lens_MultiMode'  # 실제 작업 디렉토리

if not REPO_ROOT.exists():
    print('Cloning...')
    subprocess.run(['git', 'clone', '--depth', '1', REPO_URL, str(REPO_ROOT)],
                   check=True)
else:
    print('Repo exists, pulling latest...')
    subprocess.run(['git', '-C', str(REPO_ROOT), 'pull', '--ff-only'],
                   check=True)

# v0.5 주요 파일 존재 확인
for f in [
    'scripts/phase4_v0_5_round.py',
    'ml/models/heads.py',
    'ml/training/round_eval.py',
    'ml/training/physics_pairing.py',
]:
    p = PROJ_DIR / f
    print(f'{f}: {"OK" if p.exists() else "MISSING"}')

# mode3 삭제 검증
mode3_wrapper = PROJ_DIR / 'inversion' / 'mode3_wrapper.py'
print(f'mode3_wrapper.py deleted: {not mode3_wrapper.exists()}')


## Cell 4 — 환경변수 설정 + 의존성 설치


In [ ]:
import os, sys

# ── 환경변수 ─────────────────────────────────────────────────────────────────
os.environ['LENS_DATA_PATH']            = str(TRAIN_H5)
os.environ['LENS_DATA_PATH_UNFILTERED'] = str(UNFILT_H5)
os.environ['LENS_SCALER_PATH']          = str(SCALER_PKL)
os.environ['LENS_WORK_ROOT']            = str(WORK_DIR)

print('LENS_DATA_PATH           =', os.environ['LENS_DATA_PATH'])
print('LENS_DATA_PATH_UNFILTERED=', os.environ['LENS_DATA_PATH_UNFILTERED'])
print('LENS_SCALER_PATH         =', os.environ['LENS_SCALER_PATH'])
print('LENS_WORK_ROOT           =', os.environ['LENS_WORK_ROOT'])

# ── Python 경로 ──────────────────────────────────────────────────────────────
os.chdir(str(PROJ_DIR))
if str(PROJ_DIR) not in sys.path:
    sys.path.insert(0, str(PROJ_DIR))
print('cwd:', os.getcwd())

# ── 의존성 확인 ──────────────────────────────────────────────────────────────
import subprocess
r = subprocess.run(['pip', 'install', '-q', 'h5py', 'scipy', 'pyyaml'],
                   capture_output=True, text=True)
print('pip:', r.returncode)

# 빠른 import 검증
from ml.models.error_corrector import MultiModalErrorCorrector
from ml.training.round_eval import evaluate_mode1_h0_on_loader
from ml.training.physics_pairing import add_paired_physics_predictions
print('All imports OK')


## Cell 5 — 2-epoch sanity run

acceptance/leak 게이트는 10 epoch 미만이라 skip됨.


In [ ]:
ROUND_SCRIPT = str(PROJ_DIR / 'scripts' / 'phase4_v0_5_round.py')
!python {ROUND_SCRIPT} --phase train --device cuda --workers 0 --epochs 2 --bootstrap-n 0


## Cell 6 — Full run (50 epochs, bootstrap 1000)

sanity 통과 확인 후 실행. 예상 시간: ~30-60분 (T4 기준).


In [ ]:
!python {ROUND_SCRIPT} --phase train --device cuda --workers 0 --epochs 50 --bootstrap-n 1000


## Cell 7 — 결과 확인


In [ ]:
import json
logs_dir = WORK_DIR / 'logs'

# 주요 지표 요약
for fname in ['phase4_v0_5_imgres_h0_eval.json',
              'phase4_v0_5_imgres_h0_eval_unfiltered.json',
              'phase4_v0_5_infra_equivalence.json']:
    p = logs_dir / fname
    if not p.exists():
        print(f'{fname}: not found'); continue
    data = json.loads(p.read_text())
    print(f'\n== {fname} ==')
    if 'stage_b_acceptance_report' in data:
        report = data['stage_b_acceptance_report']
        print('all_pass:', report.get('all_pass_excluding_record_only'))
        print('leak_triggered:', report.get('leak_triggered'))
        for row in report.get('pass_rows', []):
            mark = '✅' if row['pass'] else ('📝' if row['pass'] is None else '❌')
            print(f"  {mark} {row['metric']}: {row['value']} (criterion: {row['criterion']})")
    elif 'best' in data:
        m = data['best'].get('mode1', {}).get('h0', {}).get('model', {})
        print('RMSE:', m.get('RMSE'), '| r:', m.get('r'))
        cal = data['best']['mode1']['log_sigma_calibration']
        print('coverage_1sigma:', cal.get('coverage_abs_residual_le_1sigma'))


## Cell 8 — Checkpoint v0.5 검증

`head1.net.0.weight` shape이 `(64, 256)`이면 v0.5 OK. v0.4는 `(64, 384)`.


In [ ]:
import torch
ckpt_path = WORK_DIR / 'checkpoints' / 'phase4_v0_5_imgres_best.pt'

if ckpt_path.exists():
    sd = torch.load(ckpt_path, map_location='cpu', weights_only=True)
    w = sd.get('head1.net.0.weight')
    par = sd.get('par_enc.net.0.weight')
    img_keys = [k for k in sd if k.startswith('img_enc.')]
    h3_keys  = [k for k in sd if k.startswith('head3.')]

    print(f'head1.net.0.weight : {w.shape}')    # 기대: torch.Size([64, 256])
    print(f'par_enc.net.0.weight: {par.shape}')  # 기대: torch.Size([256, 20])
    print(f'img_enc keys  (should be 0): {len(img_keys)}')
    print(f'head3 keys    (should be 0): {len(h3_keys)}')

    assert w is not None and w.shape == (64, 256), f'Expected (64,256), got {w.shape}'
    assert par is not None and par.shape[1] == 20, f'Expected par_enc 20-dim, got {par.shape}'
    assert len(img_keys) == 0, f'img_enc keys found: {img_keys[:3]}'
    assert len(h3_keys)  == 0, f'head3 keys found: {h3_keys[:3]}'
    print('\nv0.5 checkpoint verified ✅')
else:
    print(f'checkpoint not found: {ckpt_path}')
    print('Full run (Cell 6) 완료 후 실행하세요.')


## Cell 9 — 산출물 목록

Kaggle output (`/kaggle/working/`)에서 회수해야 할 파일 목록.


In [ ]:
artifacts = [
    WORK_DIR / 'checkpoints' / 'phase4_v0_5_imgres_best.pt',
    WORK_DIR / 'logs'        / 'phase4_v0_5_imgres_h0_eval.json',
    WORK_DIR / 'logs'        / 'phase4_v0_5_imgres_h0_eval_unfiltered.json',
    WORK_DIR / 'logs'        / 'phase4_v0_5_imgres_long_history.json',
    WORK_DIR / 'logs'        / 'phase4_v0_5_infra_equivalence.json',
]
print('== 회수 대상 산출물 ==')
for p in artifacts:
    size = f'{p.stat().st_size/1e6:.1f} MB' if p.exists() else 'not yet'
    print(f'  {"✅" if p.exists() else "⏳"} {p.name}  ({size})')
